## Prepare GWAS data

https://cloufield.github.io/gwaslab/AssignrsID/#reference-data

In [ ]:
# eval "$(conda shell.bash hook)"
# conda init
# conda activate /work/islet_cartography_scrna/scrna_cartography_gwas
# python -m ipykernel install --user --name scrna_cartography_gwas --display-name "gwas"

In [1]:
# Path and system utilities
import os                    # Operating system interface
import sys                   # System-specific parameters and functions
import glob                  # File pattern matching
from pyhere import here      # Reproducible project paths

# dataframes
import pandas as pd
import numpy as np

# GWAS
import gwaslab as gl

sys.path.append(str(here('scripts/misc')))  # Add custom script path to system
import misc as mi

In [2]:
mi.create_directories(here("data/gwas/sumstats"))
mi.create_directories(here("data/gwas/reference"))
mi.create_directories(here("data/gwas/files"))
mi.create_directories(here("data/gwas/plots"))
mi.create_directories(here("data/gwas/sumstats_processed"))
mi.create_directories(here("data/gwas/ref_magma"))
mi.create_directories(here("data/gwas/sumstats_annotated"))
mi.create_directories(here("data/gwas/gs_files"))

/work/islet_cartography_scrna/data/gwas/sumstats Directory already exists!
/work/islet_cartography_scrna/data/gwas/reference Directory already exists!
/work/islet_cartography_scrna/data/gwas/files Directory already exists!
/work/islet_cartography_scrna/data/gwas/plots Directory already exists!
/work/islet_cartography_scrna/data/gwas/sumstats_processed Directory already exists!
/work/islet_cartography_scrna/data/gwas/ref_magma Directory already exists!
/work/islet_cartography_scrna/data/gwas/sumstats_annotated Directory already exists!
/work/islet_cartography_scrna/data/gwas/gs_files Directory created successfully!


In [ ]:
# paths 
base_dir = str(here("data/gwas"))
ref_dir =  os.path.join(base_dir, "reference")
files_dir =  os.path.join(base_dir, "files")
sums_dir = os.path.join(base_dir, "sumstats")
sumsp_dir = os.path.join(base_dir, "sumstats_processed")
sumsa_dir = os.path.join(base_dir, "sumstats_annotated")

### Download reference data

In [ ]:
# hg19 (GRCh37): 
gl.download_ref("1kg_dbsnp151_hg19_auto", directory=ref_dir)
#hg38 (GRCh38): 
gl.download_ref("1kg_dbsnp151_hg38_auto", directory=ref_dir)

### Download SumStats data

- T2D GWAS was downlaoded from: https://www.diagram-consortium.org/downloads.html (access: 2025-12-19)
- Handedness was downloaded from: https://www.ebi.ac.uk/gwas/studies/GCST90013421 (access: 2025-12-19)
- Gene and reference location files were downloaded from: https://cncr.nl/research/magma/ (access: 2025-12-19)

#### Convert T2D data to correct format for magma

In [ ]:
# file paths
files = glob.glob(os.path.join(sums_dir, 't2d_suzuki/*v2.txt'))
def process_sumstats(file_path):
    # Extract just the filename for saving later
    base_name = os.path.basename(file_path).replace(".txt", ".pval")
    
    # Load sumstats
    ss = gl.Sumstats(
        file_path,
        build=19,
        sep="\t",
        chrom="Chromsome",
        pos="Position",
        ea="EffectAllele",
        nea="NonEffectAllele",
        eaf="EAF",
        ncontrol="Ncontrols",
        ncase="Ncases",
        beta="Beta",
        se="SE",
        p="Pval",
        verbose=False
    )
    
    ss.basic_check(verbose = False)
    
    # Assign rsIDs
    ss.assign_rsid(
        ref_rsid_tsv=gl.get_path("1kg_dbsnp151_hg19_auto"),
        threads=60, 
        verbose = False
    )
    
    # Calculate total N
    ss.data["N"] = ss.data["N_CASE"] + ss.data["N_CONTROL"]

    # Remove SNPs with no ID
    ss.data = ss.data[ss.data['rsID'].notna()]
    
    # Save
    out_file = os.path.join(sumsp_dir, base_name)
    ss.data.to_csv(out_file, sep="\t", index=False)
    
    print("Finished processing:", base_name)

# Apply to all files (like purrr::map)
processed_files = list(map(process_sumstats, files))

In [ ]:
# Path where your processed .pval files are saved
processed_path = os.path.join(sumsp_dir)  # adjust if needed

# Get all .pval files
files = glob.glob(os.path.join(processed_path, '*.pval'))

# Column rename mapping
rename_col = {'rsID': 'SNP'}

# Loop through files
for file_path in files:
    # Load file
    df = pd.read_csv(file_path, sep="\t")
    
    # Rename column
    df = df.rename(columns=rename_col)
    
    # Save back
    df.to_csv(file_path, sep="\t", index=False)
    
    print("Updated columns in:", os.path.basename(file_path))

#### Convert handedness data to correct format for magma - liftover from hg38 to hg19 (because I am missing reference from hg38)

In [ ]:
# file paths
files = glob.glob(os.path.join(sums_dir, 'handedness_cuellar/32989287-GCST90013421-EFO_0009902-Build38.f.tsv'))

def process_sumstats(file_path):
    # Extract just the filename for saving later
    base_name = os.path.basename(file_path).replace(".tsv", ".pval")
    
    # Load sumstats
    ss = gl.Sumstats(
        file_path,
        rsid = "variant_id",
        build=38,
        sep="\t",
        chrom="chromosome",
        ea="effect_allele",
        nea="other_allele",
        eaf="effect_allele_frequency",
        beta="beta",
        se="standard_error",
        p="p_value",
        z = "z",
        pos = "base_pair_location",
        OR = "odds_ratio",
        n = 1766671,
        verbose=False
    )
    
    ss.basic_check(verbose = False)

    # liftover from hg38 to hg19
    mysumstats.liftover(from_build="38", 
                    to_build="19",
                    remove=True)

    # Rename columns to use for magma
    rename_col = {'rsID': 'SNP'}
    
    # Remove SNPs with no ID
    ss.data = ss.data[ss.data['rsID'].notna()]
    ss.data = ss.data.rename(columns=rename_col)
    
    # Save
    out_file = os.path.join(sumsp_dir, base_name)
    ss.data.to_csv(out_file, sep="\t", index=False)
    
    print("Finished processing:", base_name)

# Apply to all files (like purrr::map)
processed_files = list(map(process_sumstats, files))

### Map SNP ids to genes

https://cloufield.github.io/GWASTutorial/09_Gene_based_analysis/#magma-introduction
https://github.com/martinjzhang/scDRS/issues/2

First, we use ancestry-specific g1000_*.bim files from the 1000 Genomes Project (phase 3) reference panel to define the genomic positions of known SNPs and assign them to genes. SNPs located within 10 kb upstream or downstream of a gene are assigned to that gene, using gene coordinates from the NCBI gene location reference. Ancestry-specific reference files are used to ensure consistency between the SNP set and the linkage disequilibrium (LD) structure used in downstream analyses.

Second, we perform a gene-based association test to evaluate whether the combined GWAS signal of SNPs assigned to each gene is significantly stronger than expected by chance. Only SNPs present in both the GWAS summary statistics and the ancestry-matched reference panel are included in the analysis. This step uses ancestry-matched reference genotypes to account for linkage disequilibrium (LD), which arises when nearby SNPs are correlated due to co-inheritance. Correcting for LD prevents correlated SNPs from being treated as independent signals, which would would otherwise inflate gene-level test statistics and deflate p-values.

- SNPs are mapped to genes based on genomic position
- GWAS p-values are converted to SNP Z-scores
- SNPs within each gene are tested jointly
- LD between SNPs is modeled using a reference panel
- Output is one Z-score and p-value per gene

In [ ]:
# Install magma first
# mkdir MAGMA
# cd MAGMA
# wget https://ctg.cncr.nl/software/MAGMA/prog/magma_v1.10.zip
# unzip magma_v1.10.zip

In [ ]:
# Run in terminal
# !/work/islet_cartography_scrna/scripts/gwas/AB_annotate_sumstats.sh

#### Create a .gs file, top 1000 genes associated with traits

In [ ]:
# run in terminal
# !islet_cartography_scrna/scripts/gwas/AC_create_gs_file_for_scdrs.sh

In [ ]:
# https://martinjzhang.github.io/scDRS/notebooks/quickstart.html